# Huấn luyện mô hình cấu trúc Transformer cơ bản
Notebook này minh họa các bước từ xử lý dữ liệu (Tokenization), tạo Dataset/DataLoader, xây dựng kiến trúc Transformer (Decoder-only), cho đến quá trình huấn luyện và sinh văn bản (Text Generation).

## 1. Import các thư viện cần thiết

In [ ]:
import re
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

## Step 2: Xây dựng Data Pipeline

### 2.1 Tiền xử lý và Tách từ (Tokenization)
Sử dụng Regular Expression (`re`) để tách câu thành các từ khóa (tokens) và giữ lại các dấu câu đơn giản mà không làm dính chữ.

In [29]:
raw_text = "Khó khăn nào rồi cũng qua đi như sau cơn mưa, trời lại sáng."

In [30]:
def tokenize(text):
    text = text.lower().strip()
    # khác với split(" ") là nó chỉ tách dựa theo khoảng trắng, lúc đó dấu câu sẽ dính vào từ, ví dụ ['Hello,', 'Transformer!']
    # bây giờ tách theo regrex thì W+ để tách từ nhưng không lấy dấu câu, [^\w\s] là lấy dấu câu
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens

tokens = tokenize(raw_text)
tokens[:30], len(tokens)

(['khó',
  'khăn',
  'nào',
  'rồi',
  'cũng',
  'qua',
  'đi',
  'như',
  'sau',
  'cơn',
  'mưa',
  ',',
  'trời',
  'lại',
  'sáng',
  '.'],
 16)

### 2.2 Xây dựng Bộ từ vựng (Vocabulary) và Special Tokens
Trong các bài toán NLP, ta cần quy định các token đặc biệt giúp mô hình hiểu cấu trúc câu:
- `<pad>`: Padding (đệm cho các câu bằng độ dài).
- `<unk>`: Unknown (từ không có trong từ điển).
- `<bos>` / `<eos>`: Bắt đầu (Begin) và Kết thúc (End) câu.

In [31]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
IGNORE_IN_CE = -100
special_tokens = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

vocab_tokens = sorted(set(tokens))
itos = special_tokens + vocab_tokens
stoi = {tok: idx for idx, tok in enumerate(itos)}

PAD_ID = stoi[PAD_TOKEN]
UNK_ID = stoi[UNK_TOKEN]
BOS_ID = stoi[BOS_TOKEN]
EOS_ID = stoi[EOS_TOKEN]

vocab_size = len(itos)
vocab_size

20

### 2.3 Mã hóa và Giải mã (Encode / Decode)
Chuyển đổi văn bản thành danh sách biến số (IDs) để mô hình có thể tính toán, và chuyển đổi ngược lại từ IDs thành văn bản đọc được.

In [32]:
def encode(text):
    toks = tokenize(text)
    ids = [BOS_ID] + [stoi.get(tok, UNK_ID) for tok in toks] + [EOS_ID]
    return ids

def decode(ids):
    return [itos[i] for i in ids]

all_ids = encode(raw_text)
all_ids[:20], decode(all_ids[:20])

([2, 8, 9, 13, 15, 6, 14, 19, 12, 16, 7, 11, 4, 18, 10, 17, 5, 3],
 ['<bos>',
  'khó',
  'khăn',
  'nào',
  'rồi',
  'cũng',
  'qua',
  'đi',
  'như',
  'sau',
  'cơn',
  'mưa',
  ',',
  'trời',
  'lại',
  'sáng',
  '.',
  '<eos>'])

### 2.4 Chunking (Cắt dữ liệu thành các đoạn nhỏ)
Để mô hình dễ dàng học, văn bản dài đầu vào sẽ được cắt thành nhiều đoạn nhỏ (chunks) theo cơ chế cửa sổ trượt (sliding window) với tham số `max_seq_len` và `stride`.

In [33]:
def make_chunks(token_ids, max_seq_len=10, stride=6):
    chunks = []
    start = 0
    while start < len(token_ids) - 1:
        end = min(start + max_seq_len, len(token_ids))
        chunk = token_ids[start:end]
        if len(chunk) >= 2:
            chunks.append(chunk)
        if end == len(token_ids):
            break
        start += stride
    return chunks

chunks = make_chunks(all_ids, max_seq_len=10, stride=6)
len(chunks), chunks[0], decode(chunks[0])

(3,
 [2, 8, 9, 13, 15, 6, 14, 19, 12, 16],
 ['<bos>', 'khó', 'khăn', 'nào', 'rồi', 'cũng', 'qua', 'đi', 'như', 'sau'])

### 2.5 Khởi tạo PyTorch Dataset

In [34]:
class TextDataset(Dataset):
    def __init__(self, chunks):
        self.chunks = chunks
        
    def __len__(self):
        return len(self.chunks)
        
    def __getitem__(self, idx):
        # Chuyển list các id thành PyTorch Tensor
        return {
            "input_ids": torch.tensor(self.chunks[idx], dtype=torch.long)
        }

dataset = TextDataset(chunks)

### 2.6 Hàm gom Batch (Collate Function)
Quy trình gom các mẫu riêng lẻ thành Tensor Batch hoàn chỉnh:
1. **Padding**: Bơm thêm `PAD_ID` để tất cả sample trong batch có cùng độ dài.
2. **Shift Token**: Chia dữ liệu input và label (Dự đoán token tiếp theo).
3. **Masking**: Tạo `key_padding_mask` để báo cho mạng Attention bỏ qua tính toán tại các vùng token đệm.


In [35]:
def collate_fn(batch):
    # Lấy danh sách các sequence từ batch
    input_seqs = [item["input_ids"] for item in batch]
    
    # Tìm chiều dài lớn nhất trong batch hiện tại để padding
    max_len = max(seq.size(0) for seq in input_seqs)
    
    padded_inputs = []
    for x in input_seqs:
        pad_len = max_len - x.size(0)
        padded_x = torch.cat([x, torch.full((pad_len,), PAD_ID, dtype=torch.long)])
        padded_inputs.append(padded_x)

    # Gom thành tensor kích thước [Batch_size, Time_steps]
    batch_seqs = torch.stack(padded_inputs)
    
    # Next Token Prediction: Dịch chuyển (shift) chuỗi
    # input_ids: bỏ token cuối cùng
    # labels: bỏ token đầu tiên (phải dự đoán token tiếp theo)
    input_ids = batch_seqs[:, :-1]
    labels = batch_seqs[:, 1:].clone()
    
    # Tạo Mask: True ở các vị trí là PAD_ID (để báo cho Attention bỏ qua)
    key_padding_mask = (input_ids == PAD_ID)
    
    # Thay thế PAD_ID trong labels thành -100
    # Trong PyTorch, CrossEntropyLoss mặc định bỏ qua class index -100 (ignore_index=-100)
    labels[labels == PAD_ID] = IGNORE_IN_CE

    return {
        "input_ids": input_ids,
        "labels": labels,
        "key_padding_mask": key_padding_mask,
    }

In [ ]:
loader = DataLoader(dataset, batch_size=3, shuffle=False, collate_fn=collate_fn)
batch = next(iter(loader))
print("input_ids:\n", batch["input_ids"])
print("\nlabels:\n", batch["labels"])
print("\nkey_padding_mask:\n", batch["key_padding_mask"])

input_ids:
 tensor([[ 2,  8,  9, 13, 15,  6, 14, 19, 12],
        [14, 19, 12, 16,  7, 11,  4, 18, 10],
        [ 4, 18, 10, 17,  5,  3,  0,  0,  0]])

labels:
 tensor([[   8,    9,   13,   15,    6,   14,   19,   12,   16],
        [  19,   12,   16,    7,   11,    4,   18,   10,   17],
        [  18,   10,   17,    5,    3, -100, -100, -100, -100]])

key_padding_mask:
 tensor([[False, False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False,  True,  True,  True]])


## Step 3: Xây dựng Kiến trúc Mô hình

In [37]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.ln_f = nn.LayerNorm(d_model)   # ← thêm vào đây
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, key_padding_mask=None, labels=None):
        B, T = input_ids.shape
        pos_ids = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)

        x = self.token_emb(input_ids) + self.pos_emb(pos_ids)

        causal_mask = torch.triu(
            torch.ones(T, T, device=input_ids.device, dtype=torch.bool),
            diagonal=1
        )

        x = self.transformer(
            x,
            mask=causal_mask,
            src_key_padding_mask=key_padding_mask
        )

        x = self.ln_f(x)            
        logits = self.lm_head(x)    # [B, T, vocab_size]

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index= IGNORE_IN_CE
            )

        return {"logits": logits, "loss": loss}


## Step 4: Cấu hình và Huấn luyện

In [ ]:
# --- khởi tạo model ---
model = TinyGPT(
    vocab_size=vocab_size,
    d_model=64,
    nhead=4,
    num_layers=2,
    max_seq_len=64
)
optimizer = AdamW(model.parameters(), lr=1e-3)

# --- training loop ---
model.train()
for epoch in range(10):
    optimizer.zero_grad()

    output = model(
        input_ids=batch["input_ids"],
        key_padding_mask=batch["key_padding_mask"],
        labels=batch["labels"]
    )

    loss = output["loss"]
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")

Epoch 1 | Loss: 3.2138
Epoch 2 | Loss: 2.1362
Epoch 3 | Loss: 1.4612
Epoch 4 | Loss: 1.0470
Epoch 5 | Loss: 0.7242
Epoch 6 | Loss: 0.5697
Epoch 7 | Loss: 0.4713
Epoch 8 | Loss: 0.3962
Epoch 9 | Loss: 0.3490
Epoch 10 | Loss: 0.3045


## Step 5: Chạy dự đoán

In [40]:
def generate(model, prompt: str, max_seq_len=10, device="cpu"):
    model.eval()
    
    # encode prompt
    toks = tokenize(prompt)
    ids = [BOS_ID] + [stoi.get(tok, UNK_ID) for tok in toks]
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)  # [1, T]
    
    with torch.no_grad():
        for _ in range(max_seq_len):
            output = model(input_ids=input_ids)
            
            # lấy logits của token cuối cùng
            next_logits = output["logits"][0, -1, :]       # [vocab_size]
            next_id = next_logits.argmax(dim=-1).item()    # greedy
            
            # dừng nếu gặp EOS
            if next_id == EOS_ID:
                break
            
            # nối token mới vào chuỗi
            input_ids = torch.cat([
                input_ids,
                torch.tensor([[next_id]], device=device)
            ], dim=1)
    
    # decode toàn bộ (bỏ BOS)
    generated_ids = input_ids[0].tolist()
    return " ".join(decode(generated_ids[1:]))  # bỏ BOS_ID ở đầu


# thử
print(generate(model, prompt="Khó Khăn", max_seq_len=10))

khó khăn nào rồi cũng qua đi như sau cơn mưa ,
